# BNP / ANP 选择性 (v3)

v2 改了三处：
1. **binder 长度固定 90**（三组可比；同时批处理才有意义）
2. **批大小 10**（上次显存只用了 3.3/40 GB，空着浪费）
3. **先量噪声底**，再跑正式批次

顺序别跳。第 6 格（噪声底）的结果决定第 7 格值不值得跑。

## 1. 硬件

In [ ]:
!nvidia-smi
import shutil; print("可用磁盘 GB: %.0f" % (shutil.disk_usage("/content").free/1e9))

## 2. 装 BoltzGen（红字无视）

In [ ]:
!pip install -q boltzgen
!boltzgen --version

## 3. 挂 Drive + 恢复权重缓存（约 6 GB）

In [ ]:
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["UV_CACHE_DIR"] = "/content/uv_cache"

BK = "/content/drive/MyDrive/boltzgen_cache/hf_cache.tar"
if pathlib.Path(BK).exists():
    print("找到备份，恢复中…")
    !mkdir -p /content/hf_cache && tar -xf "$BK" -C /content/hf_cache
    print("恢复完成")
else:
    print("没有备份，这次从网上下（约 6 GB，一两分钟）。跑完记得点最后一格备份。")

## 4. 三个靶点文件

In [ ]:
%%writefile /content/bnp.yaml
# 靶点：人 BNP-32（B 型利钠肽，心衰的主要血液标志物）
# 目标：设计一个能结合 BNP 的小蛋白
# 肽序列直接写在这里，不需要 PDB 文件（官方 binding_disordered_peptides 例子就是这么做的）
entities:
  # 要设计的结合蛋白，长度固定 90 残基（固定是为了让三组可比，同时让批处理能用）
  - protein:
      id: A
      sequence: 90

  # 靶点：人 BNP-32，固定不变
  # S P K M V Q G S G C F G R K M D R I S S S S G L G C K V L R R H
  # 1 2 3 4 5 6 7 8 9 10            16        20        26      32
  - protein:
      id: B
      sequence: SPKMVQGSGCFGRKMDRISSSSGLGCKVLRRH

constraints:
  # BNP 的 17 元环：Cys10 - Cys26 之间的二硫键
  - bond:
      atom1: [B, 10, SG]
      atom2: [B, 26, SG]


In [ ]:
%%writefile /content/anp.yaml
# 靶点：人 ANP(1-28)（心房利钠肽）
# 目标：同一套设计流程换成 ANP，用来和 BNP 的结果对比，看选择性
entities:
  # 要设计的结合蛋白，和 BNP 那边同样固定 90 残基
  - protein:
      id: A
      sequence: 90

  # 靶点：人 ANP(1-28)，固定不变
  # S L R R S S C F G G R M D R I G A Q S G L G C N S F R Y
  # 1 2 3 4 5 6 7 8 9 10          15        20  23      28
  - protein:
      id: B
      sequence: SLRRSSCFGGRMDRIGAQSGLGCNSFRY

constraints:
  # ANP 的 17 元环：Cys7 - Cys23 之间的二硫键
  - bond:
      atom1: [B, 7, SG]
      atom2: [B, 23, SG]


In [ ]:
%%writefile /content/scr.yaml
# 阴性对照：把 BNP-32 的序列打乱
# 保留：氨基酸组成完全相同、两个 Cys 仍在第 10 和第 26 位（所以 17 元环结构仍在）
# 改变：序列顺序（与 BNP 仅 3/32 位置相同）
# 用途：如果设计对这条"假靶点"也表现出选择性，说明观察到的 Δ 是噪声
entities:
  - protein:
      id: A
      sequence: 90

  - protein:
      id: B
      sequence: KLMSGGHGSCVGSFRGSKPMRIKDRCSRVLSQ

constraints:
  - bond:
      atom1: [B, 10, SG]
      atom2: [B, 26, SG]


## 5. 验证三个文件（免费）
三个都应该打印 **Total designed residues: 90**。不是 90 就停下来告诉我。

In [ ]:
!boltzgen check /content/bnp.yaml
!boltzgen check /content/anp.yaml
!boltzgen check /content/scr.yaml

## 6. 噪声底：同一个靶点跑两遍

**这是整个项目最该先做的一格。**

同样的 BNP 靶点、同样的参数，独立跑两批各 20 个。两批之间的打分差异，就是这套流程自己的随机抖动。
后面 BNP 和 ANP 之间的差异如果没有明显大过这个抖动，那所谓"选择性"就是噪声。

预计 40 个设计，开了批处理应该比 2 分钟/个快。

In [ ]:
import time; t0=time.time()
for tag in ["a", "b"]:
    !boltzgen run /content/bnp.yaml --output /content/noise_{tag} \
      --protocol protein-anything --num_designs 20 --budget 20 \
      --diffusion_batch_size 10 --use_kernels false --reuse
print("\n两批共用时 %.1f 分钟，每个设计约 %.1f 分钟" % ((time.time()-t0)/60, (time.time()-t0)/60/40))

### 6b. 看噪声底有多大

In [ ]:
import pandas as pd, glob
def load(d):
    f = glob.glob(d + "/final_ranked_designs/all_designs_metrics.csv")
    return pd.read_csv(f[0]) if f else None

a, b = load("/content/noise_a"), load("/content/noise_b")
if a is None or b is None:
    print("还没有结果")
else:
    print("两批的列名：")
    print(list(a.columns))
    num = a.select_dtypes("number").columns
    cmp = pd.DataFrame({
        "批A均值": a[num].mean(), "批B均值": b[num].mean(),
        "批A标准差": a[num].std(),
    })
    cmp["两批均值之差"] = (cmp["批A均值"] - cmp["批B均值"]).abs()
    display(cmp.round(3))
    print("\n看法：某个指标的'两批均值之差'如果和它的'批A标准差'差不多大，说明这个指标在这个样本量下不稳定。")

## 7. 正式批次：三个靶点各 60 个

**先把第 6b 格的结果发我再跑这格。** 噪声底太大的话，参数要改，跑了也是白跑。

三个靶点必须用同样的 N。预计 180 个设计，几小时——**别在睡前点这格**，断线会白烧 units。

In [ ]:
N = 60

for name in ["bnp", "anp", "scr"]:
    !boltzgen run /content/{name}.yaml --output /content/{name}_run \
      --protocol protein-anything --num_designs {N} --budget 20 \
      --diffusion_batch_size 10 --use_kernels false --reuse

## 8. 看结果

In [ ]:
import glob, pandas as pd
for name in ["bnp", "anp", "scr"]:
    fs = glob.glob("/content/%s_run/final_ranked_designs/all_designs_metrics.csv" % name)
    if not fs:
        print(name, "：还没有结果"); continue
    df = pd.read_csv(fs[0])
    print("\n==========", name, "— 共", len(df), "个设计")
    display(df.describe().round(3))

## 9. 备份权重到 Drive（约 6 GB，跑通一次点一次就够）

In [ ]:
!mkdir -p /content/drive/MyDrive/boltzgen_cache
!tar -cf /content/drive/MyDrive/boltzgen_cache/hf_cache.tar -C /content/hf_cache .
!ls -lh /content/drive/MyDrive/boltzgen_cache/